In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# https://stackoverflow.com/questions/21971449/how-do-i-increase-the-cell-width-of-the-jupyter-ipython-notebook-in-my-browser
# from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import json
import pickle
import random
from IPython.display import display, clear_output
from PIL import Image
# from pigeonXT import annotate
from collections import defaultdict
import sys
import os
from glob import glob
from pprint import pprint
import pandas as pd
import utils
import glob

In [ ]:
# load all lockespinoza eval char pickles for dual and cross encoders

# lockespinoza_mix_neg
# key = 'lockespinoza_mix_neg'
# cross_pkls = [pickle.load(open(f, 'rb')) for f in glob.glob("output/matching_results_nov24/evaluate_ckpt/2024-12-14_13:15:30_eval-2025-02-09_00:37:55_char*.pkl")]
# dual_pkls = [pickle.load(open(f, 'rb')) for f in glob.glob("output/matching_results_nov24/evaluate_ckpt/2024-12-14_13:15:30_eval-2025-02-08_23:40:16_char*.pkl")]
# gt_test_mix_neg (leviathan)
key = 'gt_test_mix_neg'
cross_pkls = [pickle.load(open(f, 'rb')) for f in glob.glob("output/matching_results_nov24/evaluate_ckpt/2024-12-14_13:15:30_eval-2025-02-10_01:14:43_char*.pkl")]
# dual_pkls = [pickle.load(open(f, 'rb')) for f in glob.glob("output/matching_results_nov24/evaluate_ckpt/2024-12-14_13:15:30_eval-2025-02-10_01:15:28_char*.pkl")]
dual_pkls = [pickle.load(open(f, 'rb')) for f in glob.glob("output/matching_results_nov24/evaluate_ckpt/2024-12-14_13:15:30*.pkl")]


In [ ]:
def combine_results_pkls(pkls):
    pkls_comb = {
        k: {c: v for pkl in pkls for c, v in pkl[k].items()}
        for k in pkls[0].keys()
    }
    return pkls_comb

cross_pkls_comb = combine_results_pkls(cross_pkls)
dual_pkls_comb = combine_results_pkls(dual_pkls)

print(cross_pkls_comb.keys())
chars = list(cross_pkls_comb[f'{key}_knns'].keys())
print(chars)

print(dual_pkls_comb.keys())
print(dual_pkls_comb[f'{key}_knns'].keys())


In [ ]:
cross_pkls_comb[f'{key}_knns']['H']['results']['data/leviathan_matching_test_set_preprocessed/H/test/anon_12539984_62954_55height_leviathanornamentsG-0356_page1rline21_char49_H_uc.tif']['knn_paths']

In [ ]:
# load damage_score df
damage_csvs = [
    "damage_classifier_output/evaluate/gt_test_mix_neg_damagepredictions.csv",
    "damage_classifier_output/evaluate/gt_test_lockespinoza_mix_neg_damagepredictions.csv"
]
damage_dfs = [pd.read_csv(damage_csv, header=None) for damage_csv in damage_csvs]
damage_df = pd.concat(damage_dfs)
damage_df.columns = ["path", "damage"]
# change path to be basename
damage_df["path"] = damage_df["path"].apply(lambda x: os.path.basename(x))
# set index to path, and keep only damage column
damage_df = damage_df.set_index("path")["damage"]
print(f"Damage df shape: {damage_df.shape}")
# name first col path and second damage
damage_df

In [ ]:
for p in cross_pkls_comb[f'{key}_knns']['H']['results']:
    topk = cross_pkls_comb[f'{key}_knns']['H']['results'][p]['knn_paths']
    print(p, topk[0])
    print(damage_df[os.path.basename(p)], damage_df[os.path.basename(topk[0])])
    # print(len(topk))

In [ ]:
# to access hit_positions for char H
print(cross_pkls_comb[f'{key}_recall_results']['N'][-1][1000]['hit_positions'])
# to access total_potential_matches for char H
print(cross_pkls_comb[f'{key}_recall_results']['N'][-1][1000]['total_potential_matches'])

In [ ]:
# flatten out hit_positions lists across all chars

def unnest_lists(l):
    # flatten lists recursively
    return [item for sublist in l for item in (unnest_lists(sublist) if isinstance(sublist, list) else [sublist])]

print([cross_pkls_comb[f'{key}_recall_results'][c][-1][1000]['hit_positions'] for c in chars])
cross_hit_positions = unnest_lists([cross_pkls_comb[f'{key}_recall_results'][c][-1][1000]['hit_positions'] for c in chars])
print(cross_hit_positions)

print([dual_pkls_comb[f'{key}_recall_results'][c][-1][1000]['hit_positions'] for c in chars])
dual_hit_positions = unnest_lists([dual_pkls_comb[f'{key}_recall_results'][c][-1][1000]['hit_positions'] for c in chars])
print(dual_hit_positions)



In [ ]:
# plot histogram of hit_positions for cross vs dual aggregated across all chars
print(f'cross: {len(cross_hit_positions)}, median: {np.median(cross_hit_positions)}, mean: {np.mean(cross_hit_positions)}, std: {np.std(cross_hit_positions)}')
print(f'dual: {len(dual_hit_positions)}, median: {np.median(dual_hit_positions)}, mean: {np.mean(dual_hit_positions)}, std: {np.std(dual_hit_positions)}')
# plt.hist(cross_hit_positions, bins=100, alpha=0.5, label='cross')
# plt.hist(dual_hit_positions, bins=100, alpha=0.5, label='dual')
plt.violinplot([cross_hit_positions, dual_hit_positions], showmeans=True, showmedians=True, showextrema=True)
# plt.legend(loc='upper right')
labels = ['cross', 'dual']
plt.xticks([1, 2], labels)
plt.ylabel('hit positions')
plt.title('hit positions')
plt.show()

In [ ]:
# also just plot histogram of hit_positions for cross and dual
bins = np.linspace(min(min(cross_hit_positions), min(dual_hit_positions)), max(max(cross_hit_positions), max(dual_hit_positions)), 30)
# plt.hist(cross_hit_positions, alpha=0.5, bins=bins, label='cross', density=True, edgecolor='black')
# plt.hist(dual_hit_positions, alpha=0.5, bins=bins, label='dual', density=True, edgecolor='black')
# plt.legend(loc='upper right')
# plt.ylabel('hit positions')
# plt.title('hit positions')
# plt.show()

data1 = cross_hit_positions
data2 = dual_hit_positions
# Compute histograms
hist1, edges = np.histogram(data1, bins=bins, density=False)
hist2, _ = np.histogram(data2, bins=bins, density=False)

# Compute bin centers for plotting
bin_centers = (edges[:-1] + edges[1:]) / 2

# Plot histograms
plt.figure(figsize=(10, 6))

# Regular histogram
plt.bar(bin_centers, hist1, width=np.diff(edges), alpha=0.5, label='cross', color='blue', edgecolor='black')

# Upside-down histogram
plt.bar(bin_centers, -hist2, width=np.diff(edges), alpha=0.5, label='dual', color='red', edgecolor='black')

# Labels and legend
plt.xlabel('ranked similarity position (smaller better)')
plt.ylabel('count')
plt.title(f'Comparing dual and cross encoders hit positions for {key} matching\nlen(cross): {len(cross_hit_positions)}, len(dual): {len(dual_hit_positions)}')
plt.axhline(0, color='black', linewidth=1)  # Add a baseline at y=0
# Determine max value for symmetric y-axis
y_max = max(hist1.max(), hist2.max()) * 1.1 # Add 10% buffer potentially?
# Set symmetric y-limits
plt.ylim(-y_max, y_max)

# Set symmetric y-ticks
yticks = np.linspace(0, y_max, 5)  # Define evenly spaced y-ticks
plt.yticks(
    np.concatenate([-yticks[::-1], yticks]),  # Mirror ticks symmetrically
    [f"{int(t)}" for t in np.concatenate([yticks[::-1], yticks])]  # Keep labels positive
)
plt.legend()
plt.grid(True)

# Show plot
plt.show()



In [ ]:
damage_thresholds = [ 1e-10, 1e-5, 0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99, 0.999]

for c in chars:
    for p in cross_pkls_comb[f'{key}_knns'][c]['results']:
        topk = cross_pkls_comb[f'{key}_knns'][c]['results'][p]['knn_paths']
        print(p, topk[0])
        print(damage_df[os.path.basename(p)], damage_df[os.path.basename(topk[0])])
        for dt in damage_thresholds:
            # filter out hit positions below dt
            if damage_df[os.path.basename(p)] < dt:
                pass